# ЛР1. Цифровой двойник, этап 1: двигатель из данных
Цель: разработка модели поверхности намагничивания ВИД по данным заторможенного ротора и исследование влияния физических ограничений на корректность нейросетевой модели.

Этап А выполняется на синтетических данных, где известен точный ответ. Этап Б выполняется на осциллограммах стенда.

In [ ]:
# Подготовка среды: папки scripts, autograder и detective должны лежать рядом с notebooks
# (в Colab: загрузите архив материалов дисциплины и распакуйте его в /content)
import sys, os, numpy as np, matplotlib.pyplot as plt
for p in ("../scripts", "scripts", "/content/scripts", "../detective", "/content/detective"):
    if os.path.isdir(p): sys.path.insert(0, os.path.abspath(p))
from srm_model import SRM, simulate_time, cycle_angle_domain
mot = SRM(); deg = np.deg2rad
print("Модель загружена: ВИД", f"{mot.Ns}/{mot.Nr}", "Udc =", mot.Udc, "В")

## Этап А. Нейросеть без ограничений и ФИНС

In [ ]:
import lab1_pinn_demo as demo
w_nn = demo.train(0.0, seed=0)   # без физических ограничений
w_pi = demo.train(1.0, seed=0)   # ФИНС
for name, w in (("ИНС", w_nn), ("ФИНС", w_pi)):
    print(name, {k: round(float(v), 4) for k, v in demo.evaluate(w).items()})

In [ ]:
i = np.linspace(0, 20, 200); th = deg(22.5)
plt.plot(i, mot.psi(th, i), 'k', label='модель')
plt.plot(i, demo.net(w_nn, np.full_like(i, th), i), '--', label='ИНС')
plt.plot(i, demo.net(w_pi, np.full_like(i, th), i), label='ФИНС')
plt.axvspan(0, 3, alpha=0.15, color='r'); plt.xlabel('Ток, А'); plt.ylabel('ψ, Вб'); plt.legend(); plt.grid(alpha=.3)

**Задание.** Проверьте требования 1–3 ЛР1 и объясните, почему нейросеть без ограничений им не удовлетворяет. Измените коэффициент физической части функции потерь (второй аргумент `train`) и опишите его влияние.

## Подготовка файла для автопроверки

In [ ]:
import pandas as pd, pathlib
grid_path = next(p for p in ("../autograder/lab1_grid.csv", "autograder/lab1_grid.csv", "/content/autograder/lab1_grid.csv") if os.path.exists(p))
g = pd.read_csv(grid_path)
out = pathlib.Path("results/lab1"); out.mkdir(parents=True, exist_ok=True)
g["psi_Wb"] = demo.net(w_pi, g.theta_rad.values, g.i_A.values)
g.to_csv(out / "lab1_predictions.csv", index=False)
print("Сохранено:", out / "lab1_predictions.csv")

## Этап Б. Данные стенда
Загрузите осциллограммы стенда (файлы выдает преподаватель). Для каждого углового положения рассчитайте потокосцепление
$\psi(t)=\int_0^t (u - R\,i)\,d\tau$ и постройте кривые $\psi(i)$. Оцените влияние погрешности сопротивления и смещения нуля датчика тока.

In [ ]:
# ЗАДАНИЕ: замените синтетический пример на чтение файла стенда
t = np.linspace(0, 0.02, 2001); dt = t[1] - t[0]
u = np.where(t < 0.01, 300.0, -300.0)          # пример: импульс напряжения
psi_true, i_meas = np.zeros_like(t), np.zeros_like(t)
for k in range(1, len(t)):
    psi_true[k] = max(psi_true[k-1] + (u[k-1] - mot.R*i_meas[k-1])*dt, 0)
    i_meas[k] = mot.current(np.pi, psi_true[k], i_meas[k-1])
for R_err in (1.0, 1.2):
    psi_calc = np.maximum(np.cumsum((u - R_err*mot.R*i_meas))*dt, 0)
    plt.plot(i_meas, psi_calc, label=f'R × {R_err}')
plt.xlabel('Ток, А'); plt.ylabel('ψ, Вб'); plt.legend(); plt.grid(alpha=.3)